# Đọc kết quả model: khi nào "đúng 78%" là chưa đủ

**Bối cảnh.** Đội Data Science gửi Head of Customer Care một model churn: *"Decision tree sâu 3 tầng, accuracy 77.7%
trên tập test."* Head of Customer Care định dùng nó để chọn khách cho đội gọi giữ chân và hỏi bạn: *"Con số này tốt
không? Model bắt được bao nhiêu khách sắp rời? Có hơn cách chúng ta đang làm không?"* Cách đang làm là một quy tắc:
gọi khách hợp đồng month-to-month đã gắn bó tối đa 12 tháng.

Dữ liệu: bộ `telco-churn` (IBM, 7 043 khách, 26.5% rời bỏ). Bạn viết ba hàm đọc kết quả bất kỳ model nào (chỉ cần nhãn
thật và nhãn/điểm dự đoán), rồi dùng chúng để so model với hai baseline: "luôn đoán ở lại" và quy tắc đang dùng.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; không dùng `sklearn.metrics` trong ba hàm (tự đếm để hiểu).
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train model, in kết quả; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/14-ml-bigdata-for-analysts/exercises/basic/02-reading-model-results -q`. Xong thì làm phần **trắc nghiệm** trong app.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

raw = pd.read_csv(ensure("telco-churn") / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
y = (raw.Churn == "Yes").astype(int)
print(raw.shape, f"churn {y.mean():.1%}")

In [ ]:
import numpy as np
import pandas as pd

## 1. Confusion matrix

`confusion_table(y_true, y_pred) -> DataFrame`: bảng 2 × 2 số đếm (kiểu `int`), index tên `actual` gồm `0`, `1`,
cột tên `predicted` gồm `0`, `1` (1 = rời bỏ). Luôn đủ 4 ô, kể cả khi model không bao giờ đoán `1`.
Đầu vào là list, numpy array hoặc Series gồm 0/1 (hoặc bool); hai đầu vào khác độ dài thì `ValueError`.

|  | predicted 0 | predicted 1 |
| --- | --- | --- |
| **actual 0** | TN | FP |
| **actual 1** | FN | TP |

In [ ]:
def confusion_table(y_true, y_pred) -> pd.DataFrame:
    """Bảng 2x2: index actual (0, 1), cột predicted (0, 1), giá trị là số đếm int."""
    # TODO: đổi hai đầu vào sang numpy int, kiểm cùng độ dài (ValueError)
    # TODO: đếm 4 ô bằng (yt == a) & (yp == p)
    raise NotImplementedError

## 2. Bảng điểm và baseline

`scorecard(y_true, y_pred) -> dict` với đúng các khoá:

| Khoá | Ý nghĩa |
| --- | --- |
| `n` | số khách (`int`) |
| `accuracy` | (TP + TN) / n |
| `baseline_accuracy` | accuracy của model "luôn đoán lớp đông nhất" trên chính dữ liệu này |
| `precision` | TP / (TP + FP); bằng `0.0` khi model không gắn cờ ai |
| `recall` | TP / (TP + FN); bằng `0.0` khi không có ca dương nào |
| `flagged_share` | tỉ lệ khách bị gắn cờ: (TP + FP) / n |

Dùng `confusion_table`. Mọi giá trị trừ `n` là `float`.

In [ ]:
def scorecard(y_true, y_pred) -> dict:
    """n, accuracy, baseline_accuracy, precision, recall, flagged_share."""
    # TODO: cm = confusion_table(y_true, y_pred); tn, fp, fn, tp = ...
    # TODO: baseline = tỉ trọng lớp đông nhất (0 hoặc 1) trong y_true
    # TODO: chia cho 0 thì trả 0.0
    raise NotImplementedError

In [ ]:
# explore: hai baseline trên toàn bộ 7 043 khách
always_stay = np.zeros(len(y), dtype=int)
rule = ((raw.Contract == "Month-to-month") & (raw.tenure <= 12)).astype(int)
print("luôn đoán ở lại:", {k: round(v, 4) for k, v in scorecard(y, always_stay).items()})
print("quy tắc đang dùng:", {k: round(v, 4) for k, v in scorecard(y, rule).items()})
print(confusion_table(y, rule))

# Trong một phân khúc ít churn, accuracy của 'luôn đoán ở lại' cao tới mức nào?
two_year = raw.Contract == "Two year"
print("hợp đồng hai năm:", {k: round(v, 4) for k, v in scorecard(y[two_year], always_stay[two_year]).items()})

## 3. Precision và recall theo ngưỡng

Model trả **điểm** (xác suất); nhãn 0/1 chỉ có sau khi chọn ngưỡng. `threshold_table(y_true, score, thresholds)
-> DataFrame`: một dòng cho mỗi ngưỡng (giữ thứ tự đầu vào), cột đúng thứ tự `threshold`, `flagged` (số khách gắn
cờ, `int`), `tp` (`int`), `precision`, `recall`, `accuracy`. Gắn cờ khi `score >= threshold`. Dùng `scorecard`.

In [ ]:
def threshold_table(y_true, score, thresholds) -> pd.DataFrame:
    """Mỗi ngưỡng một dòng: threshold, flagged, tp, precision, recall, accuracy."""
    # TODO: với mỗi t: pred = (score >= t).astype(int); s = scorecard(y_true, pred)
    # TODO: flagged = số khách gắn cờ, tp = số khách gắn cờ đúng
    raise NotImplementedError

In [ ]:
# explore: dựng lại model của đội DS (decision tree sâu 3) với cùng cách chia train/test
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

X = pd.DataFrame({
    "contract_one_year": (raw.Contract == "One year").astype(int),
    "contract_two_year": (raw.Contract == "Two year").astype(int),
    "tenure_years": raw.tenure / 12,
    "fiber_optic": (raw.InternetService == "Fiber optic").astype(int),
    "no_internet": (raw.InternetService == "No").astype(int),
    "electronic_check": (raw.PaymentMethod == "Electronic check").astype(int),
    "monthly_charges_10": raw.MonthlyCharges / 10,
    "senior": raw.SeniorCitizen,
    "tech_support": (raw.TechSupport == "Yes").astype(int),
    "paperless": (raw.PaperlessBilling == "Yes").astype(int),
})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(export_text(tree, feature_names=list(X.columns), show_weights=True))

tree_pred = tree.predict(X_test)
print("tree, ngưỡng 0.5:", {k: round(v, 4) for k, v in scorecard(y_test, tree_pred).items()})
print(confusion_table(y_test, tree_pred))

In [ ]:
# explore: đổi ngưỡng, và so với quy tắc trên cùng tập test
tree_score = tree.predict_proba(X_test)[:, 1]
print(threshold_table(y_test, tree_score, [0.15, 0.3, 0.5, 0.7]).round(3))

rule_test = ((X_test.contract_one_year == 0) & (X_test.contract_two_year == 0) & (X_test.tenure_years <= 1)).astype(int)
print("quy tắc trên test:", {k: round(v, 4) for k, v in scorecard(y_test, rule_test).items()})

Viết ra (cho chính bạn, trước khi làm trắc nghiệm):

1. Accuracy 77.7% hơn baseline bao nhiêu điểm? Ở ngưỡng mặc định, model bắt được bao nhiêu phần trăm khách sắp rời?
2. Nếu đội gọi chỉ gọi được khoảng 400 khách trong tập test, nên dùng model ở ngưỡng nào, hay dùng quy tắc?
3. Bạn cần hỏi đội DS thêm gì trước khi tin con số (chia dữ liệu thế nào, có cột nào không có sẵn lúc dự đoán)?

Xong thì mở app, làm phần **trắc nghiệm** của bài.